# Business Entity Resolution: run on Google Colab

Run the cells from top to bottom. **A CPU runtime is enough**, because nothing here uses a GPU.

| Mode | Needs | Time on free Colab (2 CPUs) |
|---|---|---|
| `inference` | test split + shipped `models/` | minutes (small data) to about 1 h (full test set) |
| `train` | train + test splits | several hours. *Runtime → Change runtime type → High-RAM* helps |

**Disconnects are fine.** With `WORK_ON_DRIVE = True`, every finished step is saved to Google Drive.
After a disconnect, run all cells again and the pipeline continues from the step that didn't finish.

## 1. Settings
Edit the fields, then run the cell.

In [ ]:
#@title Settings { display-mode: "form" }
# Where the code comes from: "github" (clone), "drive_zip" (a .zip of the repo on Drive) or "folder" (already on disk)
CODE_SOURCE = "github"  #@param ["github", "drive_zip", "folder"]
REPO_URL = "https://github.com/Ruthwik000/amazonsummerML.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
GITHUB_TOKEN = ""  #@param {type:"string"}
CODE_PATH = "/content/drive/MyDrive/amazonsummerML.zip"  #@param {type:"string"}

# The challenge data: the student_resource folder, its dataset/ folder, or a .zip of either (train/ and test/ are found automatically)
DATASET_PATH = "/content/drive/MyDrive/student_resource.zip"  #@param {type:"string"}

MODE = "inference"  #@param ["inference", "train"]
WORK_ON_DRIVE = True  #@param {type:"boolean"}
DRIVE_WORK_ROOT = "/content/drive/MyDrive/er_runs"  #@param {type:"string"}
RUN_NAME = "run1"  #@param {type:"string"}
TEAM_NAME = "team"  #@param {type:"string"}
N_THREADS = 0  #@param {type:"integer"}

## 2. Mount Google Drive

In [ ]:
import os, sys
IN_COLAB = "google.colab" in sys.modules
needs_drive = any(str(p).startswith("/content/drive") for p in (DATASET_PATH, CODE_PATH, DRIVE_WORK_ROOT))
if IN_COLAB and needs_drive and not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")
print("Colab:", IN_COLAB, "| Drive mounted:", os.path.exists("/content/drive/MyDrive"))

## 3. Get the code

In [ ]:
import os, subprocess, zipfile, glob
BASE = "/content" if IN_COLAB else os.getcwd()
REPO = os.path.join(BASE, "amazonsummerML")

def sh(cmd):
    print("$", cmd.replace(GITHUB_TOKEN, "***") if GITHUB_TOKEN else cmd)
    subprocess.run(cmd, shell=True, check=True)

if CODE_SOURCE == "github":
    url = REPO_URL.replace("https://", f"https://{GITHUB_TOKEN}@") if GITHUB_TOKEN else REPO_URL
    if os.path.exists(os.path.join(REPO, ".git")):
        sh(f"git -C {REPO} pull --ff-only origin {BRANCH}")
    else:
        sh(f"git clone --depth 1 -b {BRANCH} {url} {REPO}")
elif CODE_SOURCE == "drive_zip":
    with zipfile.ZipFile(CODE_PATH) as z:
        z.extractall(os.path.join(BASE, "_code"))
    hits = glob.glob(os.path.join(BASE, "_code", "**", "run_pipeline.py"), recursive=True)
    assert hits, "run_pipeline.py not found inside the zip"
    REPO = os.path.dirname(hits[0])
else:
    REPO = CODE_PATH
assert os.path.exists(os.path.join(REPO, "run_pipeline.py")), f"run_pipeline.py missing in {REPO} (is the latest code pushed?)"
print("code at", REPO)

## 4. Install dependencies
This takes about 30 s. No runtime restart is needed.

In [ ]:
%pip install -q -r {REPO}/requirements-colab.txt
import polars, lightgbm, rapidfuzz, numpy
print("polars", polars.__version__, "| lightgbm", lightgbm.__version__, "| rapidfuzz", rapidfuzz.__version__, "| numpy", numpy.__version__)

## 5. Check the machine and the data

In [ ]:
import os, shutil
if N_THREADS:
    os.environ["N_THREADS"] = str(N_THREADS)
ram = os.sysconf("SC_PAGE_SIZE") * os.sysconf("SC_PHYS_PAGES") / 2**30
print(f"CPUs: {os.cpu_count()}  RAM: {ram:.1f} GB  free disk: {shutil.disk_usage('/').free / 2**30:.0f} GB")
if MODE == "train" and ram < 12:
    print("WARNING: training was tested with about 7 GB peak memory, but more RAM gives headroom. Consider a High-RAM runtime.")
assert os.path.exists(DATASET_PATH), f"DATASET_PATH not found: {DATASET_PATH}"

WORK_ROOT = os.path.join(DRIVE_WORK_ROOT if WORK_ON_DRIVE else os.path.join("/content" if IN_COLAB else os.getcwd(), "er_runs"), RUN_NAME)
WORK = os.path.join(WORK_ROOT, f"work_{MODE}")
OUT = os.path.join(WORK_ROOT, "output")
os.makedirs(WORK, exist_ok=True)
print("work dir :", WORK)
print("outputs  :", OUT)

## 6. Run the pipeline
Output streams live. To resume after a disconnect, run every cell again; finished steps are skipped.
Useful extra flags: `--force` (redo everything) and `--from <step>` (redo from one step onward, e.g. `--from submission`).

In [ ]:
import subprocess, sys
cmd = [sys.executable, "-u", os.path.join(REPO, "run_pipeline.py"), "--mode", MODE, "--dataset", DATASET_PATH,
       "--work", WORK, "--out", OUT, "--package", "--team", TEAM_NAME]
p = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in p.stdout:
    print(line, end="")
if p.wait() != 0:
    raise RuntimeError("Pipeline stopped. See the message above; fix it and re-run this cell to resume.")


## 7. Results

In [ ]:
import json, os, polars as pl
m = pl.read_csv(f"{OUT}/matching_results.tsv", separator="\t", quote_char=None, infer_schema=False).fill_null("")
c = pl.read_csv(f"{OUT}/candidate_pairs.tsv", separator="\t", quote_char=None, infer_schema=False).fill_null("")
n_ids = lambda df, col: df[col].str.split(",").list.eval(pl.element().filter(pl.element() != "")).list.len()
print(f"Source-1 entities       : {m.height}")
print(f"entities with >=1 match : {(n_ids(m, 'matched_entity_ids') > 0).sum()}  ({(n_ids(m, 'matched_entity_ids') > 0).mean():.1%})")
print(f"matched pairs           : {n_ids(m, 'matched_entity_ids').sum()}")
print(f"candidates per entity   : {n_ids(c, 'candidate_entity_ids').mean():.3f}")
meta = next((p for p in (f"{WORK}/model_stage2_cf.json", f"{REPO}/models/model_stage2_cf.json") if os.path.exists(p)))
cfg = {k: v for k, v in json.load(open(meta)).items() if k != "cols"}
print("decision config         :", cfg)
m.head(10)

### Validation score (train mode only)
There's no ground truth for the test set. In `train` mode, `crossfit.py` scores the held-out 3% of Source-1
entities (V). The lines below come from that log: every decision policy tried and the one that was kept.

In [ ]:
log = f"{WORK}/logs/crossfit.log"
if os.path.exists(log):
    print("".join(l for l in open(log) if l.startswith(("thr=", "policy=", "BEST"))))
else:
    print("no crossfit log (inference mode uses the shipped models; their validation F0.5 is", cfg.get("val_f05"), ")")

## 8. Download or keep the files
If `WORK_ON_DRIVE` is on, everything is already in Drive. This cell also downloads the files to your computer.

In [ ]:
import glob
files = [f"{OUT}/matching_results.tsv"] + glob.glob(os.path.join(WORK_ROOT, "*_submission.zip"))
for f in files:
    print(f, f"{os.path.getsize(f) / 2**20:.1f} MB")
if IN_COLAB:
    from google.colab import files as colab_files
    for f in files:
        colab_files.download(f)

### Optional: score a file against ground truth
`src/evaluate.py` prints macro F0.5 (the leaderboard metric), precision, recall, singleton accuracy, per-country scores
and blocking statistics for any predictions that have a ground-truth file:
```
!cd {REPO}/src && python evaluate.py --matching <matching.tsv> --candidate <candidates.tsv> --truth <truth.tsv> --source-dir <folder with *_source1/2/3.tsv>
```